# NITK-UsoundSim — team-by-team demo

Probe: **Philips L12-4 reference model** — 128 elements, 0.30 mm pitch, 8 MHz centre (4-12 MHz range).

The simulator runs **inside this notebook**, one section per team, in pipeline order. Each section shows what the team **receives**, what it **does**, and its **output** (name, shape, units), then plots it. Nothing is loaded from `results/`.

| Team | Output handed to the next team |
|---|---|
| T1 Transducer | `element_positions` (128,) m |
| T2 Transmit Beamforming | `pulse`, `tx_delays` (128,) s, `apodization`, `transmit_field` |
| T3 Acoustic Propagation | `propagated_wavefield` (incident wave), `travel_times` (128,) s, echoes at the elements |
| T4 Tissue Interaction + RF | phantom, reflected wave, **`raw_rf`** (128, 3574) + `t_axis` |
| T5 Receive Beamforming | **`beamformed_rf`** (3118, 256) + `z_axis` |
| T6 B-mode Formation | envelope, log compression, **`bmode`** (3118, 256) in [0, 1] |
| T7 Image Reconstruction & Post-Processing | scan-converted `image` (512, 256), **`final_bmode`** |

**How to run:** kernel **Python 3.12.4 / "Python 3 (ipykernel)"** (top-right), then **Run All** (≈15 min: the speckle phantoms take ≈4 min each). `KIND` in the setup cell picks the phantom: `"cyst_lesion"` (default), `"scatterers"` or `"point_targets"`.

# ▶ Google Colab version: run the whole project step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darshilgmaniya/NITK-UsoundSim/blob/main/demo_colab.ipynb)

**How to run:** `Runtime → Run all`. Nothing else is needed.

1. **Step 0** (next cell) gets the project into Colab, using the first option that works:
   - a project folder already in `/content`;
   - a project `.zip` you uploaded to `/content` (it is unzipped);
   - otherwise it downloads the project from GitHub (`git clone`).
2. Every team section then runs in pipeline order (T1 → T7) and shows its output.
3. The **final B-mode image** is shown in the last cell and saved as `final_bmode_colab.png`.

Colab's free tier has only 2 CPUs, so the full run takes much longer than on a laptop (≈13 min on an 8-core Mac).

In [ ]:
# Step 0: get the project into Colab
import os, glob, subprocess, zipfile
from pathlib import Path

REPO = "https://github.com/darshilgmaniya/NITK-UsoundSim.git"
BASE = Path("/content") if Path("/content").exists() else Path.cwd()

def find_project():
    for d in [BASE, *sorted(BASE.iterdir())]:
        if d.is_dir() and (d / "simulator" / "config.py").exists():
            return d
    return None

project = find_project()
if project is None and glob.glob(str(BASE / "*.zip")):            # an uploaded project zip
    zip_path = sorted(glob.glob(str(BASE / "*.zip")))[0]
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE)
    print("unzipped:", zip_path)
    project = find_project()
if project is None:                                                # download from GitHub
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(BASE / "NITK-UsoundSim")], check=True)
    project = find_project()

os.chdir(project)
print("working folder:", os.getcwd())
print(sorted(os.listdir()))

## Setup — shared `config.py` (single source of truth)

In [ ]:
import sys, time
from pathlib import Path

# Find the project root (the folder that contains simulator/config.py), whatever folder the notebook runs from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulator" / "config.py").exists())
# A kernel started before the folders were reorganised may still hold modules from the old locations: drop them.
for _name, _mod in list(sys.modules.items()):
    _f = getattr(_mod, "__file__", None) or ""
    if "Medical Ultrasound" in _f and (not Path(_f).exists() or not _f.startswith(str(ROOT))):
        del sys.modules[_name]
sys.path = [p for p in sys.path if "Medical Ultrasound" not in p or Path(p).exists()]
print("Python:", sys.executable, sys.version.split()[0], "| project:", ROOT)
import cv2, numpy as np, matplotlib.pyplot as plt
from scipy.signal import hilbert

sys.path.insert(0, str(ROOT / "simulator"))      # project modules live in simulator/
import config
from tissue_phantoms import build_phantom, simulate_rf
from receive_beamforming import das_beamform
from bmode_formation import bmode_formation
from post_processing import scan_convert, postprocess, speckle_index, to_uint8
from nitk_usoundsim.acoustic_propagation import generate_pulse, forward_propagation
from tissue_phantoms import incident_wave, point_echo
from tissue_interaction import tissue_interaction
sys.path.insert(0, str(config.T2_DIR))           # teams/T2_TX_Beamforming/Usound
from src.beamformer import calculate_tx_delays, make_apodization     # T2's code

KIND = "cyst_lesion"        # or "scatterers" / "point_targets"
plt.rcParams["figure.dpi"] = 90
print(config.PROBE_NAME)
print(f"{config.N_ELEMENTS} elements | pitch {config.PITCH*1e3:.2f} mm | f0 {config.F0/1e6:g} MHz | fs {config.FS/1e6:g} MHz | c {config.C:g} m/s | "
      f"alpha {config.ALPHA_0} dB/MHz/cm | DR {config.DYNAMIC_RANGE_DB:g} dB")
print(f"imaged x {config.SCANLINE_X_MIN*1e3:+g}..{config.SCANLINE_X_MAX*1e3:+g} mm ({config.N_SCANLINES} lines), "
      f"depth {config.Z_MIN*1e3:g}-{config.Z_MAX*1e3:g} mm")

---
# T1 — Transducer
**Receives:** `N_ELEMENTS`, `PITCH` from config. **Does:** places the 128 elements of the linear array (Philips L12-4: pitch 0.30 mm, aperture 38.4 mm) (T3's `create_linear_array`, since T1's code implements a piezo-material registry, not array geometry).
**Output:** `element_positions` — shape (128,), metres, x = lateral, all at z = 0, element 0 at the most negative x.

In [ ]:
element_positions = config.get_element_positions()
EX = element_positions
print("element_positions:", EX.shape, "| x from", round(EX[0]*1e3, 3), "to", round(EX[-1]*1e3, 3), "mm")
print(f"pitch {config.PITCH*1e3:.3f} mm = {config.PITCH/config.LAMBDA:.2f} lambda | aperture {config.APERTURE*1e3:.1f} mm")
plt.figure(figsize=(9, 1.6)); plt.plot(EX*1e3, np.zeros_like(EX), "s", ms=4)
plt.yticks([]); plt.xlabel("x (mm)"); plt.title(f"T1 output: {config.N_ELEMENTS} element positions"); plt.show()

---
# T2 — Transmit Beamforming
**Receives:** `element_positions`. **Does:** transmit pulse, per-element transmit delays and apodization, and the resulting transmit field.
**Outputs:** `pulse` (T3's Gaussian pulse at the probe's 8 MHz, 3 cycles), `tx_delays` (128,) s, `apodization` (128,), `transmit_field` (beam map).
The imaging chain uses a **broadcast** transmit (all delays zero, no apodization — T3 has no per-element weight input); T2's focused delays (`calculate_tx_delays`, focus 20 mm) are shown for comparison.

In [ ]:
tp, pulse = generate_pulse(f0=config.F0)
tx_delays = config.get_tx_delays()                              # used: broadcast
tx_delays_focus = calculate_tx_delays(EX, 0.0, 0.020, config.C)  # T2, focus (0, 20) mm
apodization = make_apodization(config.N_ELEMENTS, "hann")       # T2's window (not applied in the chain)
print("pulse:", pulse.shape, "| tx_delays:", tx_delays.shape, "| T2 focused delays max", round(tx_delays_focus.max()*1e6, 3), "us")

fig, ax = plt.subplots(1, 3, figsize=(15, 3))
ax[0].plot(tp*1e6, pulse); ax[0].set(title=f"pulse ({config.F0/1e6:g} MHz)", xlabel="time (us)")
ax[1].plot(EX*1e3, tx_delays*1e6, label="broadcast (used)"); ax[1].plot(EX*1e3, tx_delays_focus*1e6, label="T2 focus 20 mm")
ax[1].set(title="tx_delays", xlabel="element x (mm)", ylabel="us"); ax[1].legend()
ax[2].plot(EX*1e3, np.ones_like(EX), label="broadcast (used)"); ax[2].plot(EX*1e3, apodization, label="T2 Hann")
ax[2].set(title="apodization", xlabel="element x (mm)"); ax[2].legend(); plt.tight_layout(); plt.show()

In [ ]:
# transmit_field: peak of the incident wave over a grid (T3 forward_propagation x element directivity, each set of delays)
t_axis = config.get_t_axis()
gx, gz = np.linspace(-0.012, 0.012, 49), np.linspace(0.002, 0.040, 39)
def transmit_field(delays):
    f = np.zeros((gz.size, gx.size))
    for i, z in enumerate(gz):
        for j, x in enumerate(gx):
            inc = incident_wave({"x": x, "z": z}, t_axis, tau_tx=delays)
            f[i, j] = np.abs(hilbert(inc)).max()
    return f
t0 = time.time(); field_b, field_f = transmit_field(tx_delays), transmit_field(tx_delays_focus)
print(f"transmit_field: {field_b.shape} grid points, {time.time()-t0:.1f} s")
fig, ax = plt.subplots(1, 2, figsize=(9, 5))
for a, f, t in zip(ax, (field_b, field_f), ("broadcast (used)", "T2 focus at 20 mm")):
    im = a.imshow(20*np.log10(f/f.max()), cmap="inferno", vmin=-30, vmax=0, aspect="auto",
                  extent=[gx[0]*1e3, gx[-1]*1e3, gz[-1]*1e3, gz[0]*1e3])
    a.set(title=f"transmit_field: {t}", xlabel="x (mm)", ylabel="z (mm)")
fig.colorbar(im, ax=ax, label="dB re own max"); plt.show()

---
# T3 — Acoustic Propagation
**Receives:** `element_positions`, `tx_delays`, `pulse`, config (c, attenuation). **Does:** forward propagation (array → point, T3's `forward_propagation` per element × element directivity) and return propagation (point → every element): time of flight d/c and attenuation 0.5 dB/MHz/cm, via T3's `forward_propagation` / `return_propagation`.
**Outputs:** `propagated_wavefield` (incident wave at the point), `travel_times` (128,) s, and the echo received on the 128 elements. Shown for one test point at (0, 20) mm.

In [ ]:
test_point = {"x": 0.0, "z": 0.020, "amp": 1.0}
_, d_tx, travel_times = forward_propagation(test_point, EX, t_axis, f0=config.F0, tau_tx=tx_delays)  # T3's travel times
propagated_wavefield = incident_wave(test_point, t_axis, tau_tx=tx_delays)   # with element directivity
echo_at_elements = point_echo(test_point, t_axis)                            # T3 return x directivity
print("propagated_wavefield:", propagated_wavefield.shape, "| travel_times:", travel_times.shape,
      f"{travel_times.min()*1e6:.2f}-{travel_times.max()*1e6:.2f} us | echo_at_elements:", echo_at_elements.shape)
m = (t_axis > 11e-6) & (t_axis < 20e-6)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(t_axis[m]*1e6, propagated_wavefield[m]); ax[0].set(title="incident wave at (0, 20) mm", xlabel="time (us)")
ax[1].plot(EX*1e3, travel_times*1e6, "."); ax[1].set(title="travel_times (one way)", xlabel="element x (mm)", ylabel="us")
e = (t_axis > 24e-6) & (t_axis < 33e-6); lim = np.abs(echo_at_elements).max()
ax[2].imshow(echo_at_elements[:, e].T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
             extent=[0.5, config.N_ELEMENTS + 0.5, t_axis[e][-1]*1e6, t_axis[e][0]*1e6])
ax[2].set(title=f"echo received on {config.N_ELEMENTS} elements", xlabel="element #", ylabel="time (us)"); plt.tight_layout(); plt.show()

---
# T4 — Tissue Interaction + Echo/RF Generation
**Receives:** T3's propagation functions, phantom settings from config. **Does:** builds the phantom (`build_phantom`; speckle and cyst via T4's `phantom.py`), scales each incident wave by the scatterer's reflectivity (`tissue_interaction`), sends every echo back to the 64 elements and sums them (`simulate_rf`).
**Output:** `raw_rf` — shape (128, 3574) = elements × time samples, with `t_axis` (starts at −0.75 µs, one pulse half-width at 8 MHz — every later delay lookup must use this real axis).

In [ ]:
phantom = build_phantom(KIND)
sx = np.array([s["x"] for s in phantom])*1e3; sz = np.array([s["z"] for s in phantom])*1e3
amp = np.array([s["amp"] for s in phantom])
s0 = max(phantom, key=lambda s: abs(s["amp"]))
inc0 = incident_wave(s0, t_axis, tau_tx=tx_delays)
reflected = tissue_interaction(inc0, s0)
print(f"phantom {KIND}: {len(phantom)} scatterers | reflected wave example: amp {s0['amp']:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw=dict(width_ratios=[1, 2]))
ax[0].scatter(sx, sz, s=30 if len(phantom) < 10 else 0.2, c=np.abs(amp), cmap="viridis")
ax[0].invert_yaxis(); ax[0].set(title=f"phantom: {KIND}", xlabel="x (mm)", ylabel="z (mm)")
k = np.abs(inc0) > 1e-3 * np.abs(inc0).max(); k = slice(np.argmax(k) - 20, len(k) - np.argmax(k[::-1]) + 20)
ax[1].plot(t_axis[k]*1e6, inc0[k], label="incident"); ax[1].plot(t_axis[k]*1e6, reflected[k], "--", label="reflected")
ax[1].set(title="tissue_interaction at one scatterer", xlabel="time (us)"); ax[1].legend(); plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
raw_rf, t_axis = simulate_rf(phantom)
print(f"raw_rf: {raw_rf.shape} (elements x samples), t_axis from {t_axis[0]*1e6:.3f} us, {time.time()-t0:.1f} s")
lim = np.percentile(np.abs(raw_rf), 99.9)
plt.figure(figsize=(7, 5)); plt.imshow(raw_rf.T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
                                      extent=[0.5, config.N_ELEMENTS + 0.5, t_axis[-1]*1e6, t_axis[0]*1e6])
plt.ylim(60, 0); plt.xlabel("element #"); plt.ylabel("time (us)"); plt.title(f"T4 output: {config.N_ELEMENTS}-channel raw_rf"); plt.show()

---
# T5 — Receive Beamforming (DAS)
**Receives:** `raw_rf`, `t_axis`, `element_positions`, scan-line positions. **Does:** for every scan line and depth, computes the round-trip delay to each element, reads each channel with `np.interp(tau, t_axis, rf)` on the **real** time axis, applies a Hann window and sums (`das_beamform`, dynamic receive focus).
**Output:** `beamformed_rf` — shape (3118, 256) = depth samples × scan lines, with `z_axis` (0-60 mm).

In [ ]:
x_lines = config.get_scanline_positions()
beamformed_rf, z_axis = das_beamform(raw_rf, EX, x_lines, config, t_axis=t_axis)
print("beamformed_rf:", beamformed_rf.shape, "| z_axis", z_axis.shape, f"0-{z_axis[-1]*1e3:.0f} mm")
lim = np.percentile(np.abs(beamformed_rf), 99.9)
plt.figure(figsize=(4, 6)); plt.imshow(beamformed_rf, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
                                      extent=[x_lines[0]*1e3, x_lines[-1]*1e3, z_axis[-1]*1e3, z_axis[0]*1e3])
plt.ylim(40, 0); plt.xlabel("x (mm)"); plt.ylabel("z (mm)"); plt.title("T5 output: beamformed_rf"); plt.show()

---
# T6 — B-mode Formation
**Receives:** `beamformed_rf`. **Does:** envelope by Hilbert transform along depth, `20·log10(envelope / max)`, clip to the 50 dB dynamic range, map to [0, 1] (`bmode_formation`).
**Output:** `bmode` — shape (3118, 256), values 0 (≤ −50 dB) … 1 (0 dB); the raw B-mode array handed to T7.

In [ ]:
envelope = np.abs(hilbert(beamformed_rf, axis=0))
log_env = 20*np.log10(np.maximum(envelope/envelope.max(), 1e-12))
bmode = bmode_formation(beamformed_rf)
print("envelope:", envelope.shape, "| bmode:", bmode.shape, f"range [{bmode.min():.2f}, {bmode.max():.2f}]")
li = np.argmin(np.abs(x_lines)); zm = (z_axis > 0.017) & (z_axis < 0.023)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1]))
ax[0].plot(z_axis[zm]*1e3, beamformed_rf[zm, li], lw=0.7, label="beamformed RF"); ax[0].plot(z_axis[zm]*1e3, envelope[zm, li], lw=2, label="envelope")
ax[0].set(title="envelope detection (centre line)", xlabel="depth (mm)"); ax[0].legend()
ax[1].plot(z_axis[zm]*1e3, log_env[zm, li]); ax[1].axhline(-config.DYNAMIC_RANGE_DB, ls="--", c="gray")
ax[1].set(title="log compression (dB)", xlabel="depth (mm)", ylim=(-80, 3))
ax[2].imshow(bmode, cmap="gray", vmin=0, vmax=1, aspect="auto"); ax[2].set(title="T6 output: raw bmode array", xlabel="scan line #", ylabel="depth sample #")
plt.tight_layout(); plt.show()

---
# T7 — Image Reconstruction & Post-Processing
**Receives:** `bmode`, scan-line positions, `z_axis`. **Does:** (1) scan conversion — straight rectangular resample for a linear array (no fan), 0-40 mm, square pixels (`scan_convert`); (2) despeckling with the guided filter chosen in the Post-Image Processing notebook, r = 4, eps = 0.001 (`postprocess`).
**Outputs:** `image` (512, 256) at 0.078 mm/px, and **`final_bmode`** (512, 256) in [0, 1].

In [ ]:
image, x_img, z_img = scan_convert(bmode, x_lines, z_axis)
final_bmode = postprocess(image)
extent = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
print("image:", image.shape, f"pixel {(x_img[1]-x_img[0])*1e3:.3f} mm | final_bmode:", final_bmode.shape)
if KIND != "point_targets":
    print(f"speckle index: before {speckle_index(to_uint8(image)):.3f}, after {speckle_index(to_uint8(final_bmode)):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(7, 7))
for a, im, t in zip(ax, (image, final_bmode), ("scan-converted", "post-processed")):
    a.imshow(im, cmap="gray", vmin=0, vmax=1, extent=extent); a.set(title=t, xlabel="x (mm)", ylabel="z (mm)")
plt.tight_layout(); plt.show()

### T7 — seeing the filter's effect
The change is small on the full image, so it is shown four ways: zoom, difference map (x10), brightness along one row, and a stronger filter (eps 0.01) for comparison — **not used**, because it lowers edge sharpness to ≈0.78 (the notebook's rule requires ≥ 0.90). Uses the cyst image (built here if `KIND` is not `"cyst_lesion"`, ≈4 min).

In [ ]:
if KIND == "cyst_lesion":
    image_c, x_img_c, z_img_c = image, x_img, z_img
else:
    t0 = time.time()
    rf_c, t_c = simulate_rf(build_phantom("cyst_lesion"))
    bf_c, z_c = das_beamform(rf_c, EX, x_lines, config, t_axis=t_c)
    image_c, x_img_c, z_img_c = scan_convert(bmode_formation(bf_c), x_lines, z_c)
    print(f"cyst image built in {time.time()-t0:.1f} s")
_keep = (image, x_img, z_img)
image, x_img, z_img = image_c, x_img_c, z_img_c
strong = postprocess(image, eps=0.01)            # stronger setting, NOT used (edges drop to ~0.78)
chosen = postprocess(image)                      # pipeline setting: r = 4, eps = 0.001
ext = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
zx = ((x_img*1e3 >= -9) & (x_img*1e3 <= 1)); zz = ((z_img*1e3 >= 22) & (z_img*1e3 <= 38))
zext = [x_img[zx][0]*1e3, x_img[zx][-1]*1e3, z_img[zz][-1]*1e3, z_img[zz][0]*1e3]
row = np.argmin(np.abs(z_img - 0.020))

fig = plt.figure(figsize=(14, 13))
gs = fig.add_gridspec(3, 4, height_ratios=[1.25, 1, 0.55])
titles = ["before filter", "after filter (used: eps 0.001)", "stronger filter (eps 0.01, NOT used)"]
for j, (im, t) in enumerate(zip((image, chosen, strong), titles)):
    a = fig.add_subplot(gs[0, j]); a.imshow(im, cmap="gray", vmin=0, vmax=1, extent=ext)
    a.add_patch(plt.Rectangle((zext[0], zext[3]), zext[1]-zext[0], zext[2]-zext[3], fill=False, ec="yellow", lw=1.5))
    a.set(title=t, xlabel="x (mm)", ylabel="z (mm)")
    b = fig.add_subplot(gs[1, j]); b.imshow(im[np.ix_(zz, zx)], cmap="gray", vmin=0, vmax=1, extent=zext, interpolation="nearest")
    b.set(title="zoom (yellow box)", xlabel="x (mm)", ylabel="z (mm)")
a = fig.add_subplot(gs[0, 3]); d = np.abs(chosen - image)
im_ = a.imshow(np.clip(10*d, 0, 1), cmap="magma", vmin=0, vmax=1, extent=ext)
a.set(title="|after - before| x10\n(where the filter acted)", xlabel="x (mm)"); fig.colorbar(im_, ax=a, fraction=0.05)
b = fig.add_subplot(gs[1, 3]); b.imshow(np.clip(10*d[np.ix_(zz, zx)], 0, 1), cmap="magma", vmin=0, vmax=1, extent=zext, interpolation="nearest")
b.set(title="difference, zoom", xlabel="x (mm)")
c = fig.add_subplot(gs[2, :])
for im, lab, st in ((image, "before", "-"), (chosen, "after (eps 0.001)", "-"), (strong, "eps 0.01 (not used)", ":")):
    c.plot(x_img*1e3, im[row], st, lw=1.3, label=lab)
c.set(title=f"Brightness along one row (z = {z_img[row]*1e3:.1f} mm)", xlabel="x (mm)", ylabel="brightness"); c.legend(ncol=3)
plt.tight_layout(); plt.show()

u0 = to_uint8(image)
for nm, im in (("after, eps 0.001 (used)", chosen), ("eps 0.01 (not used)", strong)):
    print(f"{nm:24s}: speckle index {speckle_index(u0):.3f} -> {speckle_index(to_uint8(im)):.3f}")

image, x_img, z_img = _keep

---
# FINAL IMAGE

In [ ]:
plt.figure(figsize=(4, 8))
plt.imshow(final_bmode, cmap="gray", vmin=0, vmax=1, extent=extent)
plt.colorbar(label=f"brightness (0 = -{config.DYNAMIC_RANGE_DB:g} dB, 1 = 0 dB)", fraction=0.08)
plt.xlabel("lateral x (mm)"); plt.ylabel("depth z (mm)"); plt.title(f"Final B-mode: {KIND}"); plt.show()

### Handover summary — what each team produced in this run

In [ ]:
rows = [("T1", "element_positions", element_positions, "m"),
        ("T2", "pulse", pulse, "a.u."), ("T2", "tx_delays", tx_delays, "s"), ("T2", "transmit_field", field_b, "a.u. (grid)"),
        ("T3", "propagated_wavefield", propagated_wavefield, "a.u. vs t_axis"), ("T3", "travel_times", travel_times, "s"),
        ("T4", "raw_rf", raw_rf, "a.u., elements x samples"), ("T4", "t_axis", t_axis, "s"),
        ("T5", "beamformed_rf", beamformed_rf, "a.u., depth x lines"), ("T5", "z_axis", z_axis, "m"),
        ("T6", "bmode", bmode, "0..1 (50 dB)"), ("T7", "image", image, "0..1, px"), ("T7", "final_bmode", final_bmode, "0..1, px")]
print(f"{'team':5s} {'output':22s} {'shape':14s} units")
for team, name, arr, unit in rows:
    print(f"{team:5s} {name:22s} {str(np.shape(arr)):14s} {unit}")

---
## Bonus — all three required phantoms (≈100 s)

In [ ]:
def make_image(kind):
    rf, t = simulate_rf(build_phantom(kind))
    bf, z = das_beamform(rf, EX, x_lines, config, t_axis=t)
    img, xi, zi = scan_convert(bmode_formation(bf), x_lines, z)
    return postprocess(img), xi, zi

fig, axes = plt.subplots(1, 3, figsize=(12, 8))
for a, kind in zip(axes, ["point_targets", "scatterers", "cyst_lesion"]):
    t0 = time.time(); img, xi, zi = make_image(kind)
    print(f"{kind}: {time.time()-t0:.1f} s")
    a.imshow(img, cmap="gray", vmin=0, vmax=1, extent=[xi[0]*1e3, xi[-1]*1e3, zi[-1]*1e3, zi[0]*1e3])
    a.set(title=kind, xlabel="x (mm)", ylabel="z (mm)")
plt.suptitle(f"NITK-UsoundSim final images ({config.DYNAMIC_RANGE_DB:g} dB)"); plt.tight_layout(); plt.show()

---
# ✅ Final output: B-mode image (after post-processing)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 8))
ax[0].imshow(image, cmap="gray", vmin=0, vmax=1, extent=extent); ax[0].set(title="B-mode (before filter)", xlabel="x (mm)", ylabel="z (mm)")
ax[1].imshow(final_bmode, cmap="gray", vmin=0, vmax=1, extent=extent); ax[1].set(title="Final image (after post-processing)", xlabel="x (mm)", ylabel="z (mm)")
plt.suptitle(f"NITK-UsoundSim final output: {KIND}, {config.PROBE_NAME}"); plt.tight_layout()
plt.savefig("final_bmode_colab.png", dpi=150); plt.show()
print("saved:", Path("final_bmode_colab.png").resolve())